# AMC 2026 – Round 3 · Notebook 1: the Bloomberg data
Financial Modeling (USI) · TA session 6


**What this notebook does.** It reproduces the worked example from the TA session: it reads the Bloomberg data for *my* screen ("AMC R3 VMQ"), cleans it, checks it, and saves it as CSV files for Notebook 2. All the files it needs are already included, so you can run it without a terminal.

**To use it with your own screen**, on the Bloomberg terminal:
1. Build your screen in **EQS** and save it under a name.
2. Backtest it in **EQBT** and export the monthly returns to Excel.
3. Open `r3_bloomberg.xlsx`, type your screen name in **Settings**, cell B4. The Screen and Prices sheets now pull *your* stocks.
4. Paste your EQBT returns into the **Backtest** sheet.
5. Refresh, wait until no cell shows "Requesting Data", and save a copy as values.
6. Point this notebook to your files and run it.

### Input files

| File | Where it comes from | What it contains |
|---|---|---|
| `eqbt_export.xlsx` | EQBT, Export to Excel | Monthly returns of the strategy (V+M+Q) and of the Russell 1000 |
| `eqbt_V.xlsx`, `eqbt_M.xlsx`, `eqbt_Q.xlsx` | EQBT, one run per factor | Monthly returns of each factor alone |
| `r3_bloomberg.xlsx`, sheet **Screen** | `BEQS` | The stocks in your screen, with name and last price |
| sheet **Prices** | `BDH`, monthly | 61 month-end prices of the screen stocks |
| sheet **Other** | `BDH`, monthly | SAA ETFs, Round 2 stocks, 1-month T-bill (`GB1M Index`) |
| sheet **Live** | `BDP` | Last price of the ETFs and the Round 2 stocks |

### Output files

| CSV | Contents | Built from |
|---|---|---|
| `r3_backtest.csv` | Monthly backtest returns | EQBT files |
| `r3_holdings.csv` | Stocks in the portfolio, with last price | Screen, Live |
| `r3_prices.csv` | Monthly prices of all assets | Prices, Other |
| `r3_rf.csv` | Monthly risk-free rate | Other (T-bill) |

### Round 3 in one box (full recap at the top of notebook 02)
- **Task:** add a **quantitative strategy** (Class 6) and, later, a macro strategy (Class 7) on top of the **unchanged SAA**. **Long-only**, stocks and ETFs. Keep or close each Round 2 stock, with a reason.
- **Risk budget:** ex-ante **tracking error vs the SAA ≤ 7%** for all deviations together (monthly USD returns, last 60 months, annualized). Report volatility and 1-month 5% VaR next to the SAA's.
- **Checkpoint 1 (quant):** factor and rationale → construction rule → Class 4 backtest (design, validation, risk, discount).
- **Dates:** trades in PRTU on 28–29 November with the closes of **Friday 27 November**; report by Sunday 29 November, 23:59.
- **What this notebook does:** only data. Every number here is used by notebook 02 for one of the points above.

In [6]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.4f}".format)

DATA = "Data/"                                      # subfolder with the Bloomberg files ("" if they are next to this notebook)
FILE = DATA + "r3_bloomberg_values.xlsx"           # r3_bloomberg.xlsx after the refresh, saved as values (Paste Values)
EQBT = {"V+M+Q": "eqbt_export.xlsx",                # one EQBT export per backtest: strategy name -> file
        "Value": "eqbt_V.xlsx", "Momentum": "eqbt_M.xlsx", "Quality": "eqbt_Q.xlsx"}

def to_dates(d):
    # Excel dates can arrive as dates or as serial numbers (days since 1899-12-30)
    if pd.api.types.is_datetime64_any_dtype(d):
        return d
    num = pd.to_numeric(d, errors="coerce")
    if num.notna().sum() > d.notna().sum() / 2:                        # mostly numbers -> Excel serial dates
        return pd.to_datetime(num, unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(d, errors="coerce", format="mixed")

def read_bdh(sheet):
    # One BDH per ticker: row 1 = ticker, then two columns (date, value) per ticker
    raw = pd.read_excel(FILE, sheet_name=sheet, header=None)
    out = {}
    for j in range(0, raw.shape[1] - 1, 2):
        t = raw.iat[0, j]
        if not isinstance(t, str) or not t.strip():                    # empty slot
            continue
        s = pd.Series(pd.to_numeric(raw.iloc[1:, j + 1], errors="coerce").values,
                      index=to_dates(raw.iloc[1:, j]).values).dropna()  # '#N/A ...' rows are dropped
        s = s[s.index.notna()]
        if len(s):
            out[t.replace(" US Equity", "").replace(" Equity", "")] = s
    df = pd.DataFrame(out)
    df.index = pd.DatetimeIndex(df.index) + pd.offsets.MonthEnd(0)     # label = calendar month-end
    return df.groupby(level=0).last().sort_index()

## 1. The EQS backtest
EQBT applies the screen **at each rebalancing date with the data of that date** and the **historical** members of the Russell 1000: no look-ahead and no survivorship bias (the two avoidable biases of Class 4).

Each export has four columns: `Rebalance Date` (mm/dd/yyyy), the **periodic return** of the screen (%), its turnover (not used here) and the periodic return of the **benchmark** (RIY = Russell 1000). The first row is the start date (all zeros): we drop it.

In [7]:
def read_eqbt(file):
    # one EQBT export -> DataFrame with the screen return, its turnover, the benchmark return (decimals)
    x = pd.read_excel(DATA + file)
    x.index = pd.to_datetime(x.iloc[:, 0], format="%m/%d/%Y") + pd.offsets.MonthEnd(0)   # label = calendar month-end
    x = x.iloc[:, 1:4].apply(pd.to_numeric, errors="coerce") / 100                       # % -> decimals
    x.columns = ["ret", "turnover", "bench"]
    return x.iloc[1:]                                                                     # drop the start row (all 0)

raw = {k: read_eqbt(f) for k, f in EQBT.items()}
bt  = pd.DataFrame({k: x["ret"] for k, x in raw.items()})
bt["Russell 1000"] = raw["V+M+Q"]["bench"]

# checks: same benchmark in every file, one row per month, no gaps, no missing values
same_bench = all(np.allclose(x["bench"], bt["Russell 1000"]) for x in raw.values())
gaps = (bt.index.to_series().diff().dt.days > 31).sum()
print(f"{bt.index[0]:%b %Y} -> {bt.index[-1]:%b %Y} | {len(bt)} months | gaps: {gaps} | "
      f"missing: {bt.isna().sum().sum()} | same benchmark in all files: {same_bench}")
bt.describe().T[["count", "mean", "std", "min", "max"]]

Nov 2006 -> Oct 2026 | 240 months | gaps: 0 | missing: 0 | same benchmark in all files: True


,count,mean,std,min,max
V+M+Q,240.0000,0.0097,0.0575,-0.2414,0.1766
Value,240.0000,0.0131,0.1150,-0.3815,0.6918
Momentum,240.0000,0.0119,0.0763,-0.2879,0.2124
Quality,240.0000,0.0105,0.0538,-0.1867,0.1656
Russell 1000,240.0000,0.0097,0.0450,-0.1746,0.1321


## 2. Today's portfolio: the stocks of the screen
`BEQS` returns the stocks that pass the screen **today** (in the challenge: refresh on Saturday 28 November → data of Friday 27 November).

We read the `BEQS` columns (A:H: ticker, short name, score, market cap, last close, …). ⚠️ **Wait until every `#N/A Requesting Data...` has disappeared before you copy and paste values.** 

In [8]:
scr = pd.read_excel(FILE, sheet_name="Screen", header=None).iloc[2:, :8]                  # BEQS output, from row 3
scr.columns = ["ticker", "name", "score", "mkt_cap", "px_last", "pe", "ret_ytd", "revenue"]
scr = scr[scr["ticker"].astype(str).str.endswith(" Equity")].copy()                      # drop empty and '#N/A' rows
scr["ticker"] = scr["ticker"].str.replace(" US Equity", "")
for c in ["mkt_cap", "revenue"]:                                                          # '7’372’253’953.98' -> number
    scr[c] = pd.to_numeric(scr[c].astype(str).str.replace("’", "").str.replace("'", ""), errors="coerce")
for c in ["score", "px_last", "pe", "ret_ytd"]:
    scr[c] = pd.to_numeric(scr[c], errors="coerce")

# sector: from BDP(..., "GICS_SECTOR_NAME") in column N; if it did not load, by hand (⚠️ check with DES <GO>)
bdp_sector = pd.read_excel(FILE, sheet_name="Screen", header=None).iloc[2:, 13]
SECTOR_BY_HAND = {
    "CHRD": "Energy", "DINO": "Energy", "PR": "Energy", "VLO": "Energy", "PSX": "Energy", "APA": "Energy",
    "EOG": "Energy", "DVN": "Energy", "OVV": "Energy", "COP": "Energy", "OXY": "Energy", "MTDR": "Energy",
    "MPC": "Energy", "RRC": "Energy",
    "MU": "Information Technology", "TWLO": "Information Technology", "AVT": "Information Technology",
    "SNDK": "Information Technology",
    "NUE": "Materials", "CF": "Materials", "RS": "Materials", "RGLD": "Materials",
    "NVST": "Health Care", "BIO": "Health Care",
    "ECHO": "Communication Services", "SIRI": "Communication Services",
    "DAR": "Consumer Staples", "COLM": "Consumer Discretionary", "FDX": "Industrials"}
if bdp_sector.astype(str).str.startswith("#N/A").all():
    scr["sector"] = scr["ticker"].map(SECTOR_BY_HAND).fillna("?")
else:
    scr["sector"] = bdp_sector.loc[scr.index]

live = pd.read_excel(FILE, sheet_name="Live")                        # SAA ETFs and Round 2 stocks
live.columns = ["ticker", "name", "px_last"]
live["ticker"] = live["ticker"].str.replace(" US Equity", "")

holdings = pd.concat([scr[["ticker", "name", "sector", "px_last", "score", "mkt_cap"]].assign(group="Quant"),
                      live.assign(group="SAA / Round 2", sector="")]).set_index("ticker")

# checks: ~30 stocks, each with a price and a sector; score >= 98 (the filter of the screen)
print(f"stocks in the screen today: {len(scr)} | missing price: {scr['px_last'].isna().sum()} | "
      f"unknown sector: {(scr['sector'] == '?').sum()} | score range: {scr['score'].min():.0f}-{scr['score'].max():.0f}")
display(scr["sector"].value_counts().to_frame("stocks"))
holdings

stocks in the screen today: 29 | missing price: 0 | unknown sector: 0 | score range: 98-100


,stocks
sector,
Energy,14
Information Technology,4
Materials,4
Communication Services,2
Health Care,2
Consumer Staples,1
Industrials,1
Consumer Discretionary,1


,name,sector,px_last,score,mkt_cap,group
ticker,,,,,,
CHRD,CHORD ENERGY COR,Energy,138.6200,100.0000,7372253953.9800,Quant
ECHO,ECHOSTAR CORP-A,Communication Services,104.1600,100.0000,30308285690.3900,Quant
NVST,ENVISTA HOLDINGS,Health Care,25.8400,100.0000,4143313222.3900,Quant
DINO,HF SINCLAIR CORP,Energy,98.7600,100.0000,16924616522.7300,Quant
PR,PERMIAN RESOUR-A,Energy,21.7200,100.0000,18217123756.6500,Quant
VLO,VALERO ENERGY,Energy,350.7100,100.0000,97807545540.9500,Quant
DAR,DARLING INGREDIE,Consumer Staples,51.5700,100.0000,7990001254.7700,Quant
PSX,PHILLIPS 66,Energy,250.7000,100.0000,97791139962.9900,Quant
APA,APA CORP,Energy,45.1200,100.0000,15149310586.9700,Quant


## 3. Monthly returns for the tracking error
The **total return index** reinvests dividends, as PRTU does with our settings: $R_{i,t} = TR_{i,t}/TR_{i,t-1} - 1$. Same convention as Rounds 1 and 2: **monthly returns in USD, last 60 months**. The T-bill (`GB1M Index`, % per year) becomes a monthly rate $r_f = \text{yield}/100/12$.

In [9]:
P_stocks = read_bdh("Prices")
P_other  = read_bdh("Other")
rf = (P_other.pop("GB1M Index") / 100 / 12).rename("rf")             # monthly T-bill rate, from 2006 (backtest)
prices = pd.concat([P_other, P_stocks], axis=1).loc[P_stocks.index[0]:]

# checks: 61 month-ends (60 returns); stocks with a shorter history (recent IPOs) are fine, but note them
n_obs = prices.notna().sum()
print(f"{prices.index[0]:%b %Y} -> {prices.index[-1]:%b %Y} | {prices.shape[1]} series | "
      f"shorter than {len(prices)} months: {n_obs[n_obs < len(prices)].to_dict()}")
print(f"T-bill: {rf.index[0]:%b %Y} -> {rf.index[-1]:%b %Y}, last {rf.iloc[-1] * 1200:.2f}% per year")
prices.pct_change(fill_method=None).describe().T[["count", "mean", "std", "min", "max"]].head(12)

Sep 2021 -> Oct 2026 | 38 series | shorter than 62 months: {'SNDK': 21}
T-bill: Jan 2006 -> Oct 2026, last 3.96% per year


,count,mean,std,min,max
SPY,61.0000,0.0114,0.0449,-0.0924,0.1051
EFA,61.0000,0.0082,0.0446,-0.0922,0.1317
EEM,61.0000,0.0064,0.0520,-0.1154,0.1559
IEF,61.0000,-0.0010,0.0223,-0.0473,0.0455
LQD,61.0000,-0.0003,0.0282,-0.0670,0.0757
TIP,61.0000,0.0004,0.0178,-0.0672,0.0430
GLD,61.0000,0.0140,0.0503,-0.1168,0.1227
DBC,61.0000,0.0119,0.0496,-0.0957,0.1534
GIS,61.0000,-0.0036,0.0619,-0.2194,0.1524
CHRD,61.0000,0.0200,0.1047,-0.2336,0.3259


In [10]:
bt.to_csv("r3_backtest.csv")          # EQS backtest returns (decimals)
holdings.to_csv("r3_holdings.csv")    # today's stocks + ETFs + Round 2 stocks: name, sector, last price
prices.to_csv("r3_prices.csv")        # monthly total return index (TE)
rf.to_csv("r3_rf.csv")                # monthly T-bill rate
print("saved: r3_backtest.csv, r3_holdings.csv, r3_prices.csv, r3_rf.csv")

saved: r3_backtest.csv, r3_holdings.csv, r3_prices.csv, r3_rf.csv
